# 05 — Network Tools：网络搜索与抓取

前一章解决了本地文件操作。这章给 Agent 连上网络——让它能搜索信息、抓取文档。

两个工具：

| 工具 | 功能 |
|------|------|
| `WebSearchTool` | 搜索网络，返回结果列表（标题 + 摘要 + URL） |
| `WebFetchTool` | 抓取指定 URL 的页面内容 |

这两个工具配合使用：先搜索找到相关页面，再抓取页面内容详细阅读。

---

## 学完这章你能做到

- 用 DuckDuckGo API（无需 key）做网络搜索
- 用 `httpx` 异步抓取网页，处理重定向/超时/状态码错误
- 控制返回内容大小，防止一个网页塞满整个 token 窗口
- 验证 URL 格式，防止 Agent 传入非法 URL

In [ ]:
# !pip install httpx duckduckgo-search

In [ ]:
import sys
sys.path.insert(0, "..")

from src.tool_framework import Tool, ToolKind, ToolResult, ToolRegistry
from pathlib import Path

## 1. 工具一：WebSearchTool

### 为什么用 DuckDuckGo

- **不需要 API key**：`duckduckgo-search` 库直接调用 DDG，零配置
- **无使用限制**：个人项目和课程学习足够用
- **结果质量够用**：返回标题、链接、摘要，足够 Agent 判断哪个结果值得精读

如果是生产环境，可以换成 Brave Search API、Tavily（都有免费额度）。

In [ ]:
# 先看看 duckduckgo-search 的基础用法
from duckduckgo_search import DDGS

with DDGS() as ddgs:
    results = list(ddgs.text("Python asyncio tutorial", max_results=3))

print(f"拿到 {len(results)} 条结果")
print()
for r in results:
    print(f"标题: {r['title']}")
    print(f"URL:  {r['href']}")
    print(f"摘要: {r['body'][:100]}...")
    print()

In [ ]:
import asyncio
from duckduckgo_search import DDGS


class WebSearchTool(Tool):

    @property
    def name(self) -> str:
        return "web_search"

    @property
    def description(self) -> str:
        return (
            "搜索互联网，返回相关结果的标题、链接和摘要。"
            "适合查找文档、技术资料、错误解决方案等。"
            "搜索后如需阅读完整页面内容，使用 web_fetch 抓取具体 URL。"
        )

    @property
    def kind(self) -> ToolKind:
        return ToolKind.NETWORK

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {
                            "type": "string",
                            "description": "搜索关键词。英文关键词通常搜索质量更好。"
                        },
                        "max_results": {
                            "type": "integer",
                            "description": "返回结果数量，1-20 之间，默认 5"
                        }
                    },
                    "required": ["query"]
                }
            }
        }

    def validate(self, params: dict) -> list[str]:
        errors = []
        if "query" not in params:
            errors.append("缺少必填参数: query")
        elif not params["query"].strip():
            errors.append("query 不能为空字符串")
        if "max_results" in params:
            n = params["max_results"]
            if not isinstance(n, int) or n < 1 or n > 20:
                errors.append("max_results 必须是 1-20 之间的整数")
        return errors

    async def execute(self, params: dict) -> ToolResult:
        query = params["query"].strip()
        max_results = params.get("max_results", 5)

        try:
            # DDGS 是同步的，用 asyncio.to_thread 包一下
            # 避免在异步循环里直接调同步阻塞操作
            def _search():
                with DDGS() as ddgs:
                    return list(ddgs.text(query, max_results=max_results))

            results = await asyncio.to_thread(_search)

        except Exception as e:
            return ToolResult.error(f"搜索失败: {type(e).__name__}: {e}")

        if not results:
            return ToolResult.ok(
                f"搜索 '{query}' 没有返回结果。试试换个关键词。",
                query=query,
                result_count=0,
            )

        # 格式化输出，结构清晰让 LLM 容易解析
        lines = [f"搜索 '{query}' 的结果（{len(results)} 条）：\n"]
        for i, r in enumerate(results, 1):
            lines.append(f"[{i}] {r['title']}")
            lines.append(f"    URL: {r['href']}")
            # 摘要截断到 200 字符，够判断相关性就行
            body = r.get("body", "")[:200]
            if body:
                lines.append(f"    摘要: {body}")
            lines.append("")

        return ToolResult.ok(
            "\n".join(lines),
            query=query,
            result_count=len(results),
        )


print("WebSearchTool 定义完成")

In [ ]:
# 测试搜索
tool = WebSearchTool()

result = await tool.execute({"query": "Python httpx async HTTP client tutorial", "max_results": 3})
print(result.content)

In [ ]:
# 搜索中文关键词
result = await tool.execute({"query": "asyncio 事件循环原理", "max_results": 3})
print(result.content)

## 2. 工具二：WebFetchTool

搜索拿到 URL 列表，WebFetchTool 负责抓取具体页面的完整内容。

### 几个要处理的问题

**URL 验证**：LLM 可能生成格式不对的 URL，要提前过滤。

**大页面截断**：新闻网站、文档页面可能有几万字，全部塞进 context 太贵。设置上限（默认 50KB），超过就截断，提示 LLM 用 `offset` 读后续。

**网络错误**：超时、重定向、4xx/5xx 错误都要处理，返回清晰的错误信息。

**内容清理**：很多网页有大量 HTML 标签、脚本、广告。理想情况下要解析提取正文，但这门课先返回原始文本，保持代码简单。

In [ ]:
import httpx
from urllib.parse import urlparse


def validate_url(url: str) -> str | None:
    """
    验证 URL 格式，返回错误信息（None = 合法）。
    只允许 http/https，防止 file:// 等协议绕过安全限制。
    """
    try:
        parsed = urlparse(url)
    except Exception:
        return "URL 格式无效"

    if parsed.scheme not in ("http", "https"):
        return f"只支持 http/https，不支持: {parsed.scheme}"
    if not parsed.netloc:
        return "URL 缺少域名"
    return None  # 合法


# 测试
urls = [
    "https://docs.python.org/3/library/asyncio.html",
    "http://example.com",
    "file:///etc/passwd",
    "not-a-url",
    "ftp://ftp.example.com",
]
for url in urls:
    err = validate_url(url)
    status = "✓ 合法" if err is None else f"✗ {err}"
    print(f"  {status:20s} | {url}")

In [ ]:
MAX_FETCH_CHARS = 50_000   # 单次抓取最多返回 50KB 字符
DEFAULT_TIMEOUT = 30       # 30 秒超时


class WebFetchTool(Tool):

    @property
    def name(self) -> str:
        return "web_fetch"

    @property
    def description(self) -> str:
        return (
            "抓取指定 URL 的页面内容并返回文本。"
            "适合阅读文档、查看 GitHub 源码、获取搜索结果的完整内容。"
            "只支持 http/https URL。内容超过限制时会被截断，可用 offset 读取后续部分。"
        )

    @property
    def kind(self) -> ToolKind:
        return ToolKind.NETWORK

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "要抓取的页面 URL，必须是 http 或 https"
                        },
                        "timeout": {
                            "type": "integer",
                            "description": f"超时秒数，默认 {DEFAULT_TIMEOUT}"
                        },
                        "offset": {
                            "type": "integer",
                            "description": "从第几个字符开始返回（用于读取被截断的长页面）"
                        }
                    },
                    "required": ["url"]
                }
            }
        }

    def validate(self, params: dict) -> list[str]:
        errors = []
        if "url" not in params:
            errors.append("缺少必填参数: url")
        else:
            err = validate_url(params["url"])
            if err:
                errors.append(f"URL 无效: {err}")
        if "timeout" in params:
            t = params["timeout"]
            if not isinstance(t, int) or t < 1 or t > 120:
                errors.append("timeout 必须是 1-120 之间的整数")
        return errors

    async def execute(self, params: dict) -> ToolResult:
        url = params["url"]
        timeout = params.get("timeout", DEFAULT_TIMEOUT)
        offset = params.get("offset", 0)

        headers = {
            # 部分网站会拒绝没有 User-Agent 的请求
            "User-Agent": "Mozilla/5.0 (compatible; AIAgent/1.0)",
            "Accept": "text/html,text/plain,application/json",
        }

        try:
            async with httpx.AsyncClient(
                follow_redirects=True,    # 自动跟随重定向
                timeout=timeout,
            ) as client:
                response = await client.get(url, headers=headers)

        except httpx.TimeoutException:
            return ToolResult.error(
                f"请求超时（{timeout}s）: {url}。"
                f"可以增大 timeout 参数，或换一个速度更快的源。"
            )
        except httpx.TooManyRedirects:
            return ToolResult.error(f"重定向次数过多: {url}")
        except httpx.ConnectError as e:
            return ToolResult.error(f"连接失败: {url}\n原因: {e}")
        except Exception as e:
            return ToolResult.error(f"请求异常: {type(e).__name__}: {e}")

        # 处理 HTTP 错误状态码
        if response.status_code >= 400:
            return ToolResult.error(
                f"HTTP {response.status_code}: {url}\n"
                + self._status_hint(response.status_code),
                status_code=response.status_code,
                url=url,
            )

        # 解码内容
        try:
            text = response.text
        except Exception as e:
            return ToolResult.error(f"内容解码失败: {e}")

        total_chars = len(text)

        # 处理分页（offset）
        if offset > 0:
            text = text[offset:]

        # 截断保护
        truncated = False
        if len(text) > MAX_FETCH_CHARS:
            text = text[:MAX_FETCH_CHARS]
            next_offset = offset + MAX_FETCH_CHARS
            text += (
                f"\n\n[内容已截断。页面总长度 {total_chars} 字符，"
                f"当前显示到第 {next_offset} 字符。"
                f"使用 offset={next_offset} 读取后续内容。]"
            )
            truncated = True

        return ToolResult.ok(
            text,
            url=str(response.url),  # 最终 URL（可能经过重定向）
            status_code=response.status_code,
            content_length=total_chars,
            truncated=truncated,
        )

    def _status_hint(self, code: int) -> str:
        """给 LLM 看的错误提示，帮助它决定怎么处理。"""
        hints = {
            400: "请求格式有误。",
            401: "需要认证，该页面可能需要登录。",
            403: "访问被拒绝，该页面不允许公开访问。",
            404: "页面不存在，检查 URL 是否正确。",
            429: "请求太频繁，被限速了。稍后再试。",
            500: "服务器内部错误，稍后再试。",
            503: "服务不可用，稍后再试。",
        }
        return hints.get(code, "")


print("WebFetchTool 定义完成")

In [ ]:
# 测试抓取 Python 官方文档
tool = WebFetchTool()

result = await tool.execute({"url": "https://httpx.readthedocs.io/en/latest/"})

print(f"抓取结果: success={result.success}")
print(f"元数据: status={result.metadata.get('status_code')}, "
      f"length={result.metadata.get('content_length')}, "
      f"truncated={result.metadata.get('truncated')}")
print()
print("前 500 字符:")
print(result.content[:500])

In [ ]:
# 测试错误处理
print("测试 404:")
r = await tool.execute({"url": "https://httpbin.org/status/404"})
print(f"  success={r.success}, content={r.content[:80]}")
print()

print("测试超时（设 1 秒超时抓大页面）:")
r = await tool.execute({"url": "https://httpbin.org/delay/3", "timeout": 1})
print(f"  success={r.success}, content={r.content[:100]}")
print()

print("测试非法 URL:")
r = await tool.execute({"url": "file:///etc/passwd"})
print(f"  success={r.success}, content={r.content}")

## 3. 完整流程：搜索 + 抓取 + 总结

把两个工具串起来，模拟 Agent 处理「查一下 X 是什么」这类问题的流程。

In [ ]:
from src.llm_client import LLMClient
from src.context_manager import ContextManager, build_system_prompt
import json

# 初始化
llm = LLMClient()
ctx = ContextManager(system_prompt=build_system_prompt())
registry = ToolRegistry()
registry.register(WebSearchTool())
registry.register(WebFetchTool())

print(registry)

In [ ]:
# 手动模拟一次「搜索 + 抓取」的完整交互
# （第 06 章 AgenticLoop 会自动化这个过程）

from openai import AsyncOpenAI

raw_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

user_query = "httpx 和 requests 库有什么区别？什么时候该用哪个？"
ctx.add_user_message(user_query)

print(f"用户问: {user_query}")
print("="*60)
print("第一轮：LLM 决定调用工具...")

response = await raw_client.chat.completions.create(
    model="gpt-oss:120b",
    messages=ctx.get_messages(),
    tools=registry.get_schemas(),
)

msg = response.choices[0].message
print(f"finish_reason: {response.choices[0].finish_reason}")
print(f"tool_calls: {bool(msg.tool_calls)}")

In [ ]:
# 执行工具调用（可能不止一轮）
max_rounds = 3
round_count = 0

while response.choices[0].finish_reason == "tool_calls" and round_count < max_rounds:
    round_count += 1
    msg = response.choices[0].message

    # 把 LLM 的意图记录下来
    ctx.add_assistant_message(
        content=msg.content or "",
        tool_calls=[tc.model_dump() for tc in msg.tool_calls],
    )

    # 执行每个工具
    for tc in msg.tool_calls:
        name = tc.function.name
        params = json.loads(tc.function.arguments)

        print(f"\n[Round {round_count}] 调用工具: {name}")
        print(f"  参数: {json.dumps(params, ensure_ascii=False)}")

        result = await registry.invoke(name, params)
        print(f"  结果: success={result.success}, "
              f"长度={len(result.content)} 字符")

        ctx.add_tool_result(tc.id, result.content)

    # 把工具结果发回给 LLM
    response = await raw_client.chat.completions.create(
        model="gpt-oss:120b",
        messages=ctx.get_messages(),
        tools=registry.get_schemas(),
    )

# 最终回答
final_answer = response.choices[0].message.content
print()
print("="*60)
print("最终回答:")
print(final_answer)

## 4. 进一步：HTML 内容清理（可选）

上面 WebFetchTool 返回的是原始 HTML，里面夹杂着大量标签、脚本、CSS。

可以用 `BeautifulSoup` 提取正文，但这会增加依赖。下面是一个简单的纯文本提取方案，用正则去掉 HTML 标签：

In [ ]:
import re


def strip_html_tags(html: str) -> str:
    """
    简单粗暴地去除 HTML 标签。
    不如 BeautifulSoup 精确，但零依赖，大多数场景够用。
    """
    # 去掉 <script> 和 <style> 整个块（带内容）
    html = re.sub(r"<script[^>]*>.*?</script>", "", html, flags=re.DOTALL | re.IGNORECASE)
    html = re.sub(r"<style[^>]*>.*?</style>", "", html, flags=re.DOTALL | re.IGNORECASE)
    # 去掉所有 HTML 标签
    html = re.sub(r"<[^>]+>", " ", html)
    # 清理多余空白
    html = re.sub(r"[ \t]+", " ", html)
    html = re.sub(r"\n{3,}", "\n\n", html)
    return html.strip()


# 演示效果
sample_html = """
<html><head><style>body { color: red; }</style></head>
<body>
  <h1>Python httpx</h1>
  <p>httpx is a <strong>fully featured</strong> HTTP client.</p>
  <script>console.log('tracking');</script>
  <p>It supports async requests natively.</p>
</body></html>
"""

print("原始 HTML:")
print(sample_html)
print("清理后:")
print(strip_html_tags(sample_html))

In [ ]:
# 在 WebFetchTool.execute 里加上清理（可选，在最后写入 text 之前调用）
# 判断是 HTML 再清理，API/JSON 不要清理

def maybe_strip_html(text: str, content_type: str = "") -> str:
    """如果是 HTML 就清理，否则原样返回。"""
    is_html = (
        "text/html" in content_type.lower()
        or text.strip().startswith("<!")
        or text.strip().startswith("<html")
    )
    return strip_html_tags(text) if is_html else text

print("maybe_strip_html 可以插入到 WebFetchTool.execute 的最后 text 赋值前：")
print("  content_type = response.headers.get('content-type', '')")
print("  text = maybe_strip_html(text, content_type)")

## 5. 注册所有工具，看 schema 输出

In [ ]:
# 把前两章的文件工具加进来，展示完整的工具集
# （实际上需要从 04 的 Notebook 导入，这里重新列一下）
from pathlib import Path

full_registry = ToolRegistry()

# 文件工具（第 04 章）
cwd = Path(".")
# full_registry.register(ReadFileTool(cwd))
# full_registry.register(WriteFileTool(cwd))
# full_registry.register(EditTool(cwd))
# full_registry.register(ListDirectoryTool(cwd))
# full_registry.register(GlobTool(cwd))

# 网络工具（这章）
full_registry.register(WebSearchTool())
full_registry.register(WebFetchTool())

print("当前注册的工具:")
for t in full_registry.list_tools():
    print(f"  {t.name:20s} | kind={t.kind.value:8s} | mutating={t.is_mutating()}")

## 6. 小结

| 工具 | 关键实现细节 |
|------|-------------|
| `WebSearchTool` | `DDGS` 无需 API key，`asyncio.to_thread` 包装同步调用，摘要截 200 字符 |
| `WebFetchTool` | `httpx.AsyncClient`，自动跟随重定向，50KB 截断 + `offset` 分页 |
| `validate_url` | 只允许 http/https，防止 `file://` 等协议 |
| `_status_hint` | HTTP 错误码对应提示，帮 LLM 决定重试还是放弃 |
| `strip_html_tags` | 可选：去除 HTML 标签减少 token 浪费 |

**`asyncio.to_thread` 的用法**值得记一下：当你用到一个同步阻塞库（比如 DDGS），在 `async` 函数里直接调会阻塞整个事件循环。用 `asyncio.to_thread(fn)` 把它放进线程池，不影响其他异步任务。

---

**到这里，工具系统完整了。** 前五章造好了所有基础件：

```
LLMClient  →  能和模型说话
ContextManager  →  能管历史和 token
ToolRegistry + 7个工具  →  能操作文件和网络
```

**下一章（06）：Agentic Loop**

把这些拼起来，让 Agent 自主运转：收到任务 → 决定用什么工具 → 执行 → 看结果 → 继续或结束。这是整门课最关键的一章。

In [ ]:
await raw_client.close()
await llm.close()